# Method 1 -- Zero-Shot NLI Faithfulness Detection

Evaluates `facebook/bart-large-mnli` and `MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli` on the HaluEval QA set as zero-shot entailment/contradiction detectors, plus a SummaC-Zero sentence-level variant (Laban et al., 2021). See `docs/report.pdf` Section 3.1/4.1-4.2 for the full write-up. No training is performed in this notebook.

In [ ]:
import sys, json
sys.path.insert(0, '..')

from src.data.loader import load_halueval_qa
from src.models.zero_shot import run_zero_shot_nli, run_summac_zero
from src.evaluation.metrics import compute_metrics, print_report

ds = load_halueval_qa(num_samples=10_000)
print(f'Loaded {len(ds)} examples')

## BART-MNLI (document-level zero-shot)

In [ ]:
bart_result = run_zero_shot_nli(ds, model_name='facebook/bart-large-mnli')
print_report(bart_result.labels, bart_result.predictions, title='BART-MNLI (Zero-Shot)')

## DeBERTa-v3-Large-MNLI (document-level zero-shot)

In [ ]:
deberta_result = run_zero_shot_nli(
    ds, model_name='MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli'
)
print_report(deberta_result.labels, deberta_result.predictions,
             title='DeBERTa-v3-Large-MNLI (Zero-Shot)')

## SummaC-Zero (sentence-level aggregation, BART-MNLI backbone)

In [ ]:
summac_result = run_summac_zero(ds, model_name='facebook/bart-large-mnli')
print_report(summac_result.labels, summac_result.predictions, title='BART-MNLI + SummaC-Zero')

## Save results for the cross-method comparison in `05_analysis.ipynb`

In [ ]:
results = {
    'BART-MNLI (Zero-Shot)': compute_metrics(bart_result.labels, bart_result.predictions).as_dict(),
    'DeBERTa-v3-Large-MNLI (Zero-Shot)': compute_metrics(deberta_result.labels, deberta_result.predictions).as_dict(),
    'BART-MNLI + SummaC-Zero': compute_metrics(summac_result.labels, summac_result.predictions).as_dict(),
}

with open('../outputs/tables/zero_shot_results.json', 'w') as f:
    json.dump(results, f, indent=2)
results